In [ ]:
pip install transformers torch datasets scikit-learn pandas

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score

In [ ]:
# ==========================================
# Phase 1: Data Preparation
# ==========================================
print("Downloading and preparing dataset...")

# Load the Yelp Polarity dataset (0 = Negative, 1 = Positive)
dataset = load_dataset("yelp_polarity")

# Subsample the dataset for faster training (2000 for training, 500 for validation)
# Increase these numbers if using a powerful GPU
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
val_data = dataset['test'].shuffle(seed=42).select(range(500))

# Initialize the Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

# Apply tokenization
tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)

# Rename 'label' to 'labels' as expected by the Hugging Face model
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")

# Remove the raw text column and set format to PyTorch tensors
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_train.set_format("torch")

tokenized_val = tokenized_val.remove_columns(["text"])
tokenized_val.set_format("torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# Phase 2: Model Initialization & Training
# ==========================================
print("Initializing model...")

# Load the model with 2 labels (Negative and Positive)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Define Training Arguments
training_args = TrainingArguments(
    output_dir='./canteen_bert_results',
    num_train_epochs=3,                  # 3 passes through the data
    per_device_train_batch_size=16,      # Batch size for training
    per_device_eval_batch_size=16,       # Batch size for evaluation
    warmup_steps=100,                    # Gradual learning rate warmup
    weight_decay=0.01,                   # Helps prevent overfitting
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",         # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save model at the end of each epoch
    load_best_model_at_end=True,         # Keep the best performing model
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

# Evaluate the final model on the validation set
print("Evaluating model...")
eval_results = trainer.evaluate()
print(f"Validation Accuracy: {eval_results['eval_accuracy']:.4f}")

# Save the best model and tokenizer locally
model_path = "./final_canteen_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Initializing model...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Evaluating model...


Validation Accuracy: 0.8880


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./final_canteen_model


In [ ]:
# ==========================================
# Phase 3: Testing / Inference
# ==========================================
print("\n--- Testing Custom Canteen Reviews ---")

# Put the model in evaluation mode
model.eval()

# Map the numeric labels back to human-readable categories
label_mapping = {0: "Negative", 1: "Positive"}

def predict_review(review_text):
    # Tokenize the custom text
    inputs = tokenizer(review_text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # Move inputs to the same device as the model (GPU/CPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Perform inference without calculating gradients
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the predicted class
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return label_mapping[predicted_class_id]

# Test cases
custom_reviews = [
    "The samosas today were incredibly stale and cold.",
    "The new iced coffee is actually amazing!",
    "Standard thali, nothing special but it fills you up.",
    "Found a bug in my salad. Absolutely disgusting.",
    "The canteen staff is always friendly and quick.",
    "the biryani does not taste good."
]

for review in custom_reviews:
    prediction = predict_review(review)
    print(f"Review: '{review}'\nPredicted Sentiment: {prediction}\n")


--- Testing Custom Canteen Reviews ---
Review: 'The samosas today were incredibly stale and cold.'
Predicted Sentiment: Negative

Review: 'The new iced coffee is actually amazing!'
Predicted Sentiment: Positive

Review: 'Standard thali, nothing special but it fills you up.'
Predicted Sentiment: Positive

Review: 'Found a bug in my salad. Absolutely disgusting.'
Predicted Sentiment: Negative

Review: 'The canteen staff is always friendly and quick.'
Predicted Sentiment: Positive

Review: 'the biryani does not taste good.'
Predicted Sentiment: Negative

